# EDA of combined energy demand and weather data

## Load and combine dataset

In [ ]:
# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

In [ ]:
import pandas as pd

df_energy = pd.read_csv("../data/processed/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])
df_weather_germany = pd.read_csv("../data/processed/weather_data_mean_cities_2019_2025.csv", parse_dates=['DateUTC'])

# combine energy demand data with weather data, and save the combined dataset for future use
df_energy_weather = pd.concat([df_energy, df_weather_germany], axis=1)
df_energy_weather = df_energy_weather.loc[:,~df_energy_weather.columns.duplicated()]

df_energy_weather = df_energy_weather.sort_values('DateUTC').reset_index(drop=True)

df_energy_weather.to_csv("../data/processed/energy_weather_combined_2019_2025.csv", index=False)


## Analysis of correlation between weather and energy demand

* energy demand on different weather variables
    - apparent_temperature
    - rain
    - snowfall
    - wind_speed_10m
    - shortwave_radiation

* discarded
    - precipitation
    - temperature_2m
    - weathercode

In [ ]:
# plot correlation of energy demand and weather variables using scatter plots
import seaborn as sns
import matplotlib.pyplot as plt

df_energy_weather_daily = df_energy_weather.resample('D', on='DateUTC').mean().reset_index()

plt.figure(figsize=(6, 4))

# regression line in orange, scatter points in blue with some transparency
sns.regplot(
    data=df_energy_weather_daily,
    x="apparent_temperature",
    y="EnergyDemand",
    scatter_kws={"alpha": 0.2, "color": "steelblue"},
    line_kws={"color": "orange"},
    lowess=True
)

plt.title("Correlation: Apparent Temperature vs Energy Demand daily average")
plt.xlabel("Apparent Temperature (°C)")
plt.ylabel("Energy Demand")
plt.show()


In [ ]:

# plot correlation of energy demand and weather variables using scatter plots
plt.figure(figsize=(6, 4))

# regression line in orange, scatter points in blue with some transparency
sns.regplot(
    data=df_energy_weather_daily,
    x="rain",
    y="EnergyDemand",
    scatter_kws={"alpha": 0.2, "color": "steelblue"},
    line_kws={"color": "orange"},
    lowess=True
)

plt.title("Correlation: Rain vs Energy Demand daily average")
plt.xlabel("Rain (mm)")
plt.ylabel("Energy Demand")
plt.show()

In [ ]:

# plot correlation of energy demand and weather variables using scatter plots
plt.figure(figsize=(6, 4))

# regression line in orange, scatter points in blue with some transparency
sns.regplot(
    data=df_energy_weather_daily,
    x="snowfall",
    y="EnergyDemand",
    scatter_kws={"alpha": 0.2, "color": "steelblue"},
    line_kws={"color": "orange"},
    lowess=True
)

plt.title("Correlation: Snowfall vs Energy Demand daily average")
plt.xlabel("Snowfall (mm)")
plt.ylabel("Energy Demand")
plt.show()

In [ ]:

# plot correlation of energy demand and weather variables using scatter plots
plt.figure(figsize=(6, 4))

# regression line in orange, scatter points in blue with some transparency
sns.regplot(
    data=df_energy_weather_daily,
    x="shortwave_radiation",
    y="EnergyDemand",
    scatter_kws={"alpha": 0.2, "color": "steelblue"},
    line_kws={"color": "orange"},
    lowess=True
)

plt.title("Correlation: Sunshine (Shortwave Radiation) vs Energy Demand daily average")
plt.xlabel("Sunshine (Shortwave Radiation) (W/m²)")
plt.ylabel("Energy Demand")
plt.show()

In [ ]:
# plot correlation of energy demand and weather variables using scatter plots
plt.figure(figsize=(6, 4))

# regression line in orange, scatter points in blue with some transparency
sns.regplot(
    data=df_energy_weather_daily,
    x="wind_speed_10m",
    y="EnergyDemand",
    scatter_kws={"alpha": 0.2, "color": "steelblue"},
    line_kws={"color": "orange"},
    lowess=True
)

plt.title("Correlation: Wind Speed vs Energy Demand daily average")
plt.xlabel("Wind Speed (m/s)")
plt.ylabel("Energy Demand")
plt.show()

## feature engineering

* create new features based on existing ones
* weather features 
    - rolling averages, lagged variables
    - heating and cooling degree depending on apparent temperatur
* energy demand feature
    - rolling average, lagged variables
* drop nan values (168 points) caused by lagging calculation 

In [ ]:
df_energy_weather['apparent_temperature_rolling_mean_24h'] = df_energy_weather['apparent_temperature'].shift(1).rolling(window=24).mean()
df_energy_weather['apparent_temperature_lag_24h'] = df_energy_weather['apparent_temperature'].shift(24)

# add rolling average and lagged varirable for shortwave_radiation_0m
df_energy_weather['shortwave_radiation_0m_rolling_mean_24h'] = df_energy_weather['shortwave_radiation'].shift(1).rolling(window=24).mean()
df_energy_weather['shortwave_radiation_0m_lag_24h'] =   df_energy_weather['shortwave_radiation'].shift(24)

# add heating degree days (HDD) and cooling degree days (CDD) features
base_temperature_heating = 15  # base temperature for heating degree days
base_temperature_cooling = 25  # base temperature for cooling degree days
df_energy_weather['heating_degree'] = df_energy_weather['apparent_temperature'].apply(lambda x: max(0, base_temperature_heating - x))  # HDD is calculated as the difference between a base temperature (e.g., 18°C) and the actual temperature, but only if the actual temperature is below the base temperature
df_energy_weather['cooling_degree'] = df_energy_weather['apparent_temperature'].apply(lambda x: max(0, x - base_temperature_cooling))  # CDD is calculated as the difference between the actual temperature and a base temperature (e.g., 25°C), but only if the actual temperature is above the base temperature

# add lagged features for energy demand (shifted by 24 hours, 168 hours (1 week) to capture daily, weekly, and yearly patterns)
df_energy_weather['EnergyDemand_lag_24h'] = df_energy_weather['EnergyDemand'].shift(24)   # 1 day
df_energy_weather['EnergyDemand_lag_168h'] = df_energy_weather['EnergyDemand'].shift(168)   # 1 week
#df_energy_weather['EnergyDemand_lag_8760h'] = df_energy_weather['EnergyDemand'].shift(8760) # 1 year, removed after feature importance analysis showed it was not useful

# rolling mean of past demand (shift first to avoid leakage)
df_energy_weather['EnergyDemand_rolling_mean_24h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(24).mean()   # daily pattern
df_energy_weather['EnergyDemand_rolling_mean_168h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(168).mean() # weekly pattern
#df_energy_weather['EnergyDemand_rolling_mean_8760h'] = df_energy_weather['EnergyDemand'].shift(1).rolling(8760).mean() # yearly pattern, removed after feature importance analysis showed it was not useful

# drop rows with missing values
df_energy_weather = df_energy_weather.dropna()

#display(df_energy_weather.isna().sum())

df_energy_weather.to_csv("../data/energy_weather_2019_2025.csv", index=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# change the size of the plot
plt.figure(figsize=(20, 18))
energy_weather_corr = df_energy_weather.drop('DateUTC', axis=1).corr()
sns.heatmap(energy_weather_corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Energy and Weather Variables') 
plt.show()

In [ ]:
# plot energy demand histogram
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

plt.figure(figsize=(4, 2))
sns.histplot(df_energy_weather['EnergyDemand'], bins=50, kde=True)
plt.title('Energy Demand Distribution')
plt.xlabel('Energy Demand')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

df_energy_weather = df_energy_weather.sort_values("DateUTC")
df_energy_weather_weekly = df_energy_weather.resample('W', on='DateUTC').mean().reset_index()
df_energy_weather_weekly_2024 = df_energy_weather_weekly[df_energy_weather_weekly['DateUTC'].dt.year == 2024]

fig, ax1 = plt.subplots(figsize=(10,4))

# Verbrauch links
ax1.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["EnergyDemand_rolling_mean_24h"],
    label="EnergyDemand",
    color="steelblue"
)

ax1.set_ylabel("EnergyDemand (MWh)", color="steelblue")

# Temperatur rechts
ax2 = ax1.twinx()

ax2.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["apparent_temperature_rolling_mean_24h"],
    label="ApparentTemperature",
    color="orange"
)

ax2.set_ylabel("ApparentTemperature (°C)", color="orange")
plt.grid(False)

plt.title("EnergyDemand and Apparent-Temperature weekly average in 2024")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

df_energy_weather = df_energy_weather.sort_values("DateUTC")
df_energy_weather_weekly = df_energy_weather.resample('W', on='DateUTC').mean().reset_index()
df_energy_weather_weekly_2024 = df_energy_weather_weekly[df_energy_weather_weekly['DateUTC'].dt.year == 2024]

fig, ax1 = plt.subplots(figsize=(10,4))

# Verbrauch links
ax1.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["EnergyDemand_rolling_mean_24h"],
    label="EnergyDemand",
    color="steelblue"
)

ax1.set_ylabel("EnergyDemand (MWh)", color="steelblue")

# Temperatur rechts
ax2 = ax1.twinx()

ax2.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["wind_speed_10m"],
    label="WindSpeed",
    color="orange"
)

ax2.set_ylabel("WindSpeed (m/s)", color="orange")
plt.grid(False)

plt.title("EnergyDemand and WindSpeed weekly average in 2024")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_energy_weather = pd.read_csv("../data/processed/energy_weather_2019_2025.csv", parse_dates=['DateUTC'])

df_energy_weather = df_energy_weather.sort_values("DateUTC")
df_energy_weather_weekly = df_energy_weather.resample('W', on='DateUTC').mean().reset_index()
df_energy_weather_weekly_2024 = df_energy_weather_weekly[df_energy_weather_weekly['DateUTC'].dt.year == 2024]

fig, ax1 = plt.subplots(figsize=(10,4))

# Verbrauch links
ax1.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["EnergyDemand_rolling_mean_24h"],
    label="EnergyDemand",
    color="steelblue"
)

ax1.set_ylabel("EnergyDemand (MWh)", color="steelblue")

# Temperatur rechts
ax2 = ax1.twinx()

ax2.plot(
    df_energy_weather_weekly_2024["DateUTC"],
    df_energy_weather_weekly_2024["shortwave_radiation_0m_rolling_mean_24h"],
    label="Sunshine (ShortwaveRadiation)",
    color="orange"
)

ax2.set_ylabel("ShortwaveRadiation (W/m²)", color="orange")

plt.grid(False)
plt.title("EnergyDemand and Sunshine weekly average in 2024")
plt.show()